<a href="https://colab.research.google.com/github/NovyteLabs/Emergenics/blob/main/rpzl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# RPZL-only recursive zoom model on Tiny Shakespeare
# With symbolic prime-ratio backboning (float32-safe)
# ─────────────────────────────────────────────────────────────────────────────

# 1) Setup: install + fetch
!pip install -q tqdm transformers scikit-learn sympy
!wget -q -O /content/tiny.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# 2) Imports
import math, torch
import numpy as np
from torch import nn
from tqdm import tqdm
from transformers import GPT2TokenizerFast
from sympy import primerange
from sklearn.neighbors import NearestNeighbors

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device)

# 3) Tokenizer + stream
tok = GPT2TokenizerFast.from_pretrained("gpt2", add_prefix_space=True)
vocab_size, embed_dim = tok.vocab_size, 128
embedding = nn.Embedding(vocab_size, embed_dim).to(device)

token_stream = []
with open("/content/tiny.txt", encoding="utf-8") as f:
    for line in f:
        ids = tok(line, add_special_tokens=False).input_ids
        token_stream.extend(ids)
        if len(token_stream) >= 200_000:
            break
token_stream = token_stream[:200_000]
print(f"Streamed {len(token_stream)} tokens.")

# 4) Prime-ratio setup for symbolic backbone
primes = list(primerange(2, 300))
prime_ratios = np.array(sorted({p/q for p in primes for q in primes if p <= q}))

def nearest_prime_ratio(val, ratios):
    return ratios[np.abs(ratios - val).argmin()]

# 5) RPZL modules
out_dim = 64
class RPZLEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1 = nn.Linear(embed_dim, out_dim)
        self.act = nn.Tanh()
        self.lin2 = nn.Linear(out_dim, out_dim)
    def forward(self, E):
        return self.lin2(self.act(self.lin1(E)))

class RPZLDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone_proj = nn.Linear(out_dim, out_dim)
        self.lin1 = nn.Linear(out_dim * 2, embed_dim)
        self.act = nn.Tanh()
        self.lin2 = nn.Linear(embed_dim, vocab_size)
    def forward(self, φ, symbolic_aug):
        z = self.backbone_proj(symbolic_aug)
        φ_aug = torch.cat([φ, z], dim=-1)
        return self.lin2(self.act(self.lin1(φ_aug)))

rpzl_encoder = RPZLEncoder().to(device)
rpzl_decoder = RPZLDecoder().to(device)
opt = torch.optim.Adam(
    list(embedding.parameters()) +
    list(rpzl_encoder.parameters()) +
    list(rpzl_decoder.parameters()),
    lr=5e-4
)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# 6) Recursive prime-patch with symbolic attention
PRIMES = [2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53]
def mesh_encode_with_backbone(seq_ids, window=64, stride=64, k=5):
    patches, φ_blocks = [], []
    for i in range(0, len(seq_ids) - window + 1, stride):
        ids = torch.tensor(seq_ids[i:i+window], device=device)
        E = embedding(ids)
        E_diff = E[1:] - E[:-1]
        if max(PRIMES) >= E_diff.shape[0]:
            continue
        base_patch = E_diff[PRIMES]
        φ = rpzl_encoder(base_patch).mean(0)
        φ_blocks.append(φ.detach().cpu().numpy())
        patches.append(φ)
    if not patches:
        return torch.empty(0), torch.empty(0)

    φ_arr = np.stack(φ_blocks)
    nbrs = NearestNeighbors(n_neighbors=min(k, len(φ_arr))).fit(φ_arr)
    _, indices = nbrs.kneighbors(φ_arr)

    augments = []
    for i, φi in enumerate(φ_arr):
        weights = []
        for j in indices[i]:
            diffs = np.abs(φi / (φ_arr[j] + 1e-6) - np.array([
                nearest_prime_ratio(v, prime_ratios)
                for v in φi / (φ_arr[j] + 1e-6)
            ]))
            weights.append(np.exp(-5.0 * diffs).mean())
        weights = np.array(weights)
        weights /= weights.sum()
        aug = (weights[:, None] * φ_arr[indices[i]]).sum(axis=0)
        augments.append(torch.tensor(aug, dtype=torch.float32, device=device))
    return torch.stack(patches), torch.stack(augments)

# 7) Training loop
BATCH, WINDOW, STRIDE = 32, 64, 16
def batchify(stream, bs):
    step, L = STRIDE * bs, len(stream)
    for i in range(0, L - WINDOW - step + 1, step):
        chunk = stream[i : i + step + WINDOW]
        yield [chunk[j : j + WINDOW + STRIDE] for j in range(0, step, STRIDE)]

loader = list(batchify(token_stream, BATCH))
for epoch in range(1):
    pbar = tqdm(loader, desc="Training Epoch")
    for batch in pbar:
        φs, augs, tgts = [], [], []
        for seq in batch:
            φ, symb = mesh_encode_with_backbone(seq[:-1], WINDOW, STRIDE)
            if φ.numel() == 0: continue
            φs.append(φ)
            augs.append(symb)
            targets = [seq[j + WINDOW] for j in range(0, len(seq) - WINDOW, STRIDE)]
            tgts.append(torch.tensor(targets, device=device))
        if not φs: continue
        Φb = nn.utils.rnn.pad_sequence(φs, batch_first=True).float()
        symb_b = nn.utils.rnn.pad_sequence(augs, batch_first=True).float()
        tgt = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=-100)
        opt.zero_grad()
        logits = rpzl_decoder(Φb, symb_b)
        loss = criterion(logits.view(-1, vocab_size), tgt.view(-1))
        loss.backward()
        opt.step()
        pbar.set_postfix(loss=f"{loss.item():.3f}")

# 8) Validation
with torch.no_grad():
    seq = token_stream[-(WINDOW + STRIDE + 1):-1]
    Φv, symb_v = mesh_encode_with_backbone(seq[:-1], WINDOW, STRIDE)
    if Φv.numel() > 0:
        logits = rpzl_decoder(Φv.unsqueeze(0).float(), symb_v.unsqueeze(0).float()).log_softmax(-1)[0]
        tgt = torch.tensor([seq[j + WINDOW] for j in range(0, len(seq) - WINDOW, STRIDE)], device=device)
        nll = -logits[range(tgt.size(0)), tgt].mean()
        print(f"Validation perplexity ≈ {math.exp(nll.item()):.2f}")
    else:
        print("No windows for validation.")



Running on cuda


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Streamed 200000 tokens.


Training Epoch: 100%|██████████| 390/390 [00:35<00:00, 10.97it/s, loss=6.236]


Validation perplexity ≈ 88.48


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# RPZL recursive zoom model on Tiny Shakespeare with symbolic prime backboning
# Includes: training, perplexity, and improved token-visible generation
# ─────────────────────────────────────────────────────────────────────────────

# 1) Setup
!pip install -q tqdm transformers scikit-learn sympy
!wget -q -O /content/tiny.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# 2) Imports
import math, torch
import numpy as np
from torch import nn
from tqdm import tqdm
from transformers import GPT2TokenizerFast
from sympy import primerange
from sklearn.neighbors import NearestNeighbors

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device)

# 3) Tokenizer + token stream
tok = GPT2TokenizerFast.from_pretrained("gpt2", add_prefix_space=True)
vocab_size, embed_dim = tok.vocab_size, 128
embedding = nn.Embedding(vocab_size, embed_dim).to(device)

token_stream = []
with open("/content/tiny.txt", encoding="utf-8") as f:
    for line in f:
        ids = tok(line, add_special_tokens=False).input_ids
        token_stream.extend(ids)
        if len(token_stream) >= 200_000:
            break
token_stream = token_stream[:200_000]
print(f"Streamed {len(token_stream)} tokens.")

# 4) Prime-ratio setup
primes = list(primerange(2, 300))
prime_ratios = np.array(sorted({p/q for p in primes for q in primes if p <= q}))
def nearest_prime_ratio(val, ratios):
    return ratios[np.abs(ratios - val).argmin()]

# 5) RPZL model
out_dim = 64
class RPZLEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1 = nn.Linear(embed_dim, out_dim)
        self.act = nn.Tanh()
        self.lin2 = nn.Linear(out_dim, out_dim)
    def forward(self, E):
        return self.lin2(self.act(self.lin1(E)))

class RPZLDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone_proj = nn.Linear(out_dim, out_dim)
        self.lin1 = nn.Linear(out_dim * 2, embed_dim)
        self.act = nn.Tanh()
        self.lin2 = nn.Linear(embed_dim, vocab_size)
    def forward(self, φ, symbolic_aug):
        z = self.backbone_proj(symbolic_aug)
        φ_aug = torch.cat([φ, z], dim=-1)
        return self.lin2(self.act(self.lin1(φ_aug)))

rpzl_encoder = RPZLEncoder().to(device)
rpzl_decoder = RPZLDecoder().to(device)
opt = torch.optim.Adam(
    list(embedding.parameters()) +
    list(rpzl_encoder.parameters()) +
    list(rpzl_decoder.parameters()), lr=5e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# 6) Patch encoder + symbolic backboning
PRIMES = [2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53]
def mesh_encode_with_backbone(seq_ids, window=64, stride=64, k=5):
    patches, φ_blocks = [], []
    for i in range(0, len(seq_ids) - window + 1, stride):
        ids = torch.tensor(seq_ids[i:i+window], device=device)
        E = embedding(ids)
        E_diff = E[1:] - E[:-1]
        if max(PRIMES) >= E_diff.shape[0]: continue
        base_patch = E_diff[PRIMES]
        φ = rpzl_encoder(base_patch).mean(0)
        φ_blocks.append(φ.detach().cpu().numpy())
        patches.append(φ)
    if not patches:
        return torch.empty(0), torch.empty(0)
    φ_arr = np.stack(φ_blocks)
    nbrs = NearestNeighbors(n_neighbors=min(k, len(φ_arr))).fit(φ_arr)
    _, indices = nbrs.kneighbors(φ_arr)
    augments = []
    for i, φi in enumerate(φ_arr):
        weights = []
        for j in indices[i]:
            diffs = np.abs(φi / (φ_arr[j] + 1e-6) - np.array([
                nearest_prime_ratio(v, prime_ratios)
                for v in φi / (φ_arr[j] + 1e-6)
            ]))
            weights.append(np.exp(-5.0 * diffs).mean())
        weights = np.array(weights)
        weights /= weights.sum()
        aug = (weights[:, None] * φ_arr[indices[i]]).sum(axis=0)
        augments.append(torch.tensor(aug, dtype=torch.float32, device=device))
    return torch.stack(patches), torch.stack(augments)

# 7) Training loop
BATCH, WINDOW, STRIDE = 32, 64, 16
def batchify(stream, bs):
    step, L = STRIDE * bs, len(stream)
    for i in range(0, L - WINDOW - step + 1, step):
        chunk = stream[i : i + step + WINDOW]
        yield [chunk[j : j + WINDOW + STRIDE] for j in range(0, step, STRIDE)]

loader = list(batchify(token_stream, BATCH))
for epoch in range(1):
    pbar = tqdm(loader, desc="Training Epoch")
    for batch in pbar:
        φs, augs, tgts = [], [], []
        for seq in batch:
            φ, symb = mesh_encode_with_backbone(seq[:-1], WINDOW, STRIDE)
            if φ.numel() == 0: continue
            φs.append(φ)
            augs.append(symb)
            targets = [seq[j + WINDOW] for j in range(0, len(seq) - WINDOW, STRIDE)]
            tgts.append(torch.tensor(targets, device=device))
        if not φs: continue
        Φb = nn.utils.rnn.pad_sequence(φs, batch_first=True).float()
        symb_b = nn.utils.rnn.pad_sequence(augs, batch_first=True).float()
        tgt = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=-100)
        opt.zero_grad()
        logits = rpzl_decoder(Φb, symb_b)
        loss = criterion(logits.view(-1, vocab_size), tgt.view(-1))
        loss.backward()
        opt.step()
        pbar.set_postfix(loss=f"{loss.item():.3f}")

# 8) Validation
with torch.no_grad():
    seq = token_stream[-(WINDOW + STRIDE + 1):-1]
    Φv, symb_v = mesh_encode_with_backbone(seq[:-1], WINDOW, STRIDE)
    if Φv.numel() > 0:
        logits = rpzl_decoder(Φv.unsqueeze(0).float(), symb_v.unsqueeze(0).float()).log_softmax(-1)[0]
        tgt = torch.tensor([seq[j + WINDOW] for j in range(0, len(seq) - WINDOW, STRIDE)], device=device)
        nll = -logits[range(tgt.size(0)), tgt].mean()
        print(f"Validation perplexity ≈ {math.exp(nll.item()):.2f}")
    else:
        print("No windows for validation.")

# 9) Generation (sampling + visible tokens)
def generate_text_from_seed(seed_ix=None, max_tokens=100, window=64, stride=64):
    if seed_ix is None:
        seed_ix = np.random.randint(0, len(token_stream) - (window + stride + max(PRIMES) + max_tokens))
    context = token_stream[seed_ix : seed_ix + window + stride + max(PRIMES)]
    generated = context.copy()
    print(f"🔹 Seed:\n{tok.decode(generated)}\n{'-'*50}")
    for step in range(max_tokens):
        seq = generated[-(window + stride + max(PRIMES)):]
        φv, symb_v = mesh_encode_with_backbone(seq, window, stride)
        if φv.numel() == 0:
            print(f"[{step}] ⚠️ No valid patch found. Stopping.")
            break
        logits = rpzl_decoder(φv.unsqueeze(0).float(), symb_v.unsqueeze(0).float())
        next_logits = logits[0, -1]
        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_id)
        decoded = tok.decode([next_id])
        print(f"[{step}] → {next_id} → {repr(decoded)}")
    final_output = tok.decode(generated, clean_up_tokenization_spaces=False)
    print("\\n📝 Generated Text:\\n" + "-" * 60)
    print(final_output)
    return final_output

# 🔁 Run generation
generate_text_from_seed(max_tokens=120)


Running on cuda
Streamed 200000 tokens.


Training Epoch: 100%|██████████| 390/390 [00:34<00:00, 11.23it/s, loss=6.178]


Validation perplexity ≈ 82.89
🔹 Seed:
 slain?
 Let me put in your minds, if you forget,
 What you have been ere now, and what you are;
 Withal, what I have been, and what I am.
 
 QUEEN MARGARET:
 A murderous villain, and so still thou art.
 
 GLOUCESTER:
 Poor Clarence did forsake his father, Warwick;
 Yea, and forswore himself,--which Jesu pardon!--
 
 QUEEN MARGARET:
 Which God revenge!
 
 GLOUCESTER:
 To fight on Edward's party for the crown;
 And for his meed, poor lord, he is mew'd up.
 I would to God my heart were flint, like Edward's;
 Or Edward's soft and pitiful, like mine
 I
--------------------------------------------------
[0] → 47834 → 'Yep'
[1] → 356 → ' we'
[2] → 338 → "'s"
[3] → 284 → ' to'
[4] → 262 → ' the'
[5] → 25 → ':'
[6] → 11 → ','
[7] → 892 → ' think'
[8] → 198 → '\n'
[9] → 1867 → ' What'
[10] → 618 → ' when'
[11] → 26 → ';'
[12] → 4249 → ' nor'
[13] → 8046 → ' fault'
[14] → 339 → ' he'
[15] → 198 → '\n'
[16] → 11200 → ' skept'
[17] → 198 → '\n'
[18] → 1350 → '

" slain?\n Let me put in your minds, if you forget,\n What you have been ere now, and what you are;\n Withal, what I have been, and what I am.\n \n QUEEN MARGARET:\n A murderous villain, and so still thou art.\n \n GLOUCESTER:\n Poor Clarence did forsake his father, Warwick;\n Yea, and forswore himself,--which Jesu pardon!--\n \n QUEEN MARGARET:\n Which God revenge!\n \n GLOUCESTER:\n To fight on Edward's party for the crown;\n And for his meed, poor lord, he is mew'd up.\n I would to God my heart were flint, like Edward's;\n Or Edward's soft and pitiful, like mine\n IYep we's to the:, think\n What when; nor fault he\n skept\nbe own promot\n and not\n;1999 suit 's on thee me, There what the  a and on highBeing. in-! sign\n, circuit WhereT\n\n world dis:? request counterterrorism; We,\n>(-orset; hint, isest\n giveas, thyoth' dwell\n marqu to I,\n V of soothingDrug eng then, did\x19, Sail\n gates., virtue them ever As tieate their thyish orphans and,\n use, a :"

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# RPZL-only recursive zoom model on Tiny Shakespeare
# With symbolic prime-ratio backboning (float32-safe)
# ─────────────────────────────────────────────────────────────────────────────

# 1) Setup: install + fetch
!pip install -q tqdm transformers scikit-learn sympy
!wget -q -O /content/tiny.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# 2) Imports
import math, torch
import numpy as np
from torch import nn
from tqdm import tqdm
from transformers import GPT2TokenizerFast
from sympy import primerange
from sklearn.neighbors import NearestNeighbors

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device)

# 3) Tokenizer + stream
tok = GPT2TokenizerFast.from_pretrained("gpt2", add_prefix_space=True)
vocab_size, embed_dim = tok.vocab_size, 128
embedding = nn.Embedding(vocab_size, embed_dim).to(device)

token_stream = []
with open("/content/tiny.txt", encoding="utf-8") as f:
    for line in f:
        ids = tok(line, add_special_tokens=False).input_ids
        token_stream.extend(ids)
        if len(token_stream) >= 200_000:
            break
token_stream = token_stream[:200_000]
print(f"Streamed {len(token_stream)} tokens.")

# 4) Prime-ratio setup for symbolic backbone
primes = list(primerange(2, 300))
prime_ratios = np.array(sorted({p/q for p in primes for q in primes if p <= q}))

def nearest_prime_ratio(val, ratios):
    return ratios[np.abs(ratios - val).argmin()]

# 5) RPZL modules
out_dim = 64
class RPZLEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1 = nn.Linear(embed_dim, out_dim)
        self.act = nn.Tanh()
        self.lin2 = nn.Linear(out_dim, out_dim)
    def forward(self, E):
        return self.lin2(self.act(self.lin1(E)))

class RPZLDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone_proj = nn.Linear(out_dim, out_dim)
        self.lin1 = nn.Linear(out_dim * 2, embed_dim)
        self.act = nn.Tanh()
        self.lin2 = nn.Linear(embed_dim, vocab_size)
    def forward(self, φ, symbolic_aug):
        z = self.backbone_proj(symbolic_aug)
        φ_aug = torch.cat([φ, z], dim=-1)
        return self.lin2(self.act(self.lin1(φ_aug)))

rpzl_encoder = RPZLEncoder().to(device)
rpzl_decoder = RPZLDecoder().to(device)
opt = torch.optim.Adam(
    list(embedding.parameters()) +
    list(rpzl_encoder.parameters()) +
    list(rpzl_decoder.parameters()),
    lr=5e-4
)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# 6) Recursive prime-patch with symbolic attention
PRIMES = [2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53]
def mesh_encode_with_backbone(seq_ids, window=64, stride=64, k=5):
    patches, φ_blocks = [], []
    for i in range(0, len(seq_ids) - window + 1, stride):
        ids = torch.tensor(seq_ids[i:i+window], device=device)
        E = embedding(ids)
        E_diff = E[1:] - E[:-1]
        if max(PRIMES) >= E_diff.shape[0]:
            continue
        base_patch = E_diff[PRIMES]
        φ = rpzl_encoder(base_patch).mean(0)
        φ_blocks.append(φ.detach().cpu().numpy())
        patches.append(φ)
    if not patches:
        return torch.empty(0), torch.empty(0)

    φ_arr = np.stack(φ_blocks)
    nbrs = NearestNeighbors(n_neighbors=min(k, len(φ_arr))).fit(φ_arr)
    _, indices = nbrs.kneighbors(φ_arr)

    augments = []
    for i, φi in enumerate(φ_arr):
        weights = []
        for j in indices[i]:
            diffs = np.abs(φi / (φ_arr[j] + 1e-6) - np.array([
                nearest_prime_ratio(v, prime_ratios)
                for v in φi / (φ_arr[j] + 1e-6)
            ]))
            weights.append(np.exp(-5.0 * diffs).mean())
        weights = np.array(weights)
        weights /= weights.sum()
        aug = (weights[:, None] * φ_arr[indices[i]]).sum(axis=0)
        augments.append(torch.tensor(aug, dtype=torch.float32, device=device))
    return torch.stack(patches), torch.stack(augments)

# 7) Training loop
BATCH, WINDOW, STRIDE = 32, 64, 16
def batchify(stream, bs):
    step, L = STRIDE * bs, len(stream)
    for i in range(0, L - WINDOW - step + 1, step):
        chunk = stream[i : i + step + WINDOW]
        yield [chunk[j : j + WINDOW + STRIDE] for j in range(0, step, STRIDE)]

loader = list(batchify(token_stream, BATCH))
for epoch in range(1):
    pbar = tqdm(loader, desc="Training Epoch")
    for batch in pbar:
        φs, augs, tgts = [], [], []
        for seq in batch:
            φ, symb = mesh_encode_with_backbone(seq[:-1], WINDOW, STRIDE)
            if φ.numel() == 0: continue
            φs.append(φ)
            augs.append(symb)
            targets = [seq[j + WINDOW] for j in range(0, len(seq) - WINDOW, STRIDE)]
            tgts.append(torch.tensor(targets, device=device))
        if not φs: continue
        Φb = nn.utils.rnn.pad_sequence(φs, batch_first=True).float()
        symb_b = nn.utils.rnn.pad_sequence(augs, batch_first=True).float()
        tgt = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=-100)
        opt.zero_grad()
        logits = rpzl_decoder(Φb, symb_b)
        loss = criterion(logits.view(-1, vocab_size), tgt.view(-1))
        loss.backward()
        opt.step()
        pbar.set_postfix(loss=f"{loss.item():.3f}")

# 8) Validation
with torch.no_grad():
    seq = token_stream[-(WINDOW + STRIDE + 1):-1]
    Φv, symb_v = mesh_encode_with_backbone(seq[:-1], WINDOW, STRIDE)
    if Φv.numel() > 0:
        logits = rpzl_decoder(Φv.unsqueeze(0).float(), symb_v.unsqueeze(0).float()).log_softmax(-1)[0]
        tgt = torch.tensor([seq[j + WINDOW] for j in range(0, len(seq) - WINDOW, STRIDE)], device=device)
        nll = -logits[range(tgt.size(0)), tgt].mean()
        print(f"Validation perplexity ≈ {math.exp(nll.item()):.2f}")
    else:
        print("No windows for validation.")

# 9) Generation with temperature + top-k sampling
def sample_logits(logits, temperature=0.8, top_k=50):
    if temperature <= 0:
        return torch.argmax(logits, dim=-1).item()
    logits = logits / temperature
    if top_k > 0:
        topk_vals, topk_idx = torch.topk(logits, top_k)
        probs = torch.softmax(topk_vals, dim=-1)
        return topk_idx[torch.multinomial(probs, 1)].item()
    return torch.multinomial(torch.softmax(logits, dim=-1), 1).item()

seed_text = """
Lords:
We have.

First Lord:
And grieve to hear't.
""".strip()

print("\n🔹 Seed:\n")
print(seed_text)

with torch.no_grad():
    seq = tok(seed_text, add_special_tokens=False).input_ids
    for _ in range(120):
        if len(seq) < WINDOW + STRIDE:
            pad = [tok.pad_token_id] * (WINDOW + STRIDE - len(seq))
            seq = pad + seq
        φ, symb = mesh_encode_with_backbone(seq[-(WINDOW + STRIDE):], WINDOW, STRIDE)
        if φ.numel() == 0: break
        logits = rpzl_decoder(φ.unsqueeze(0).float(), symb.unsqueeze(0).float())[0, -1]
        next_token = sample_logits(logits, temperature=0.8, top_k=50)
        seq.append(next_token)

    print("\n📝 Generated Text:\n" + "-"*60)
    print(tok.decode(seq, skip_special_tokens=True))


Running on cuda
Streamed 200000 tokens.


Training Epoch: 100%|██████████| 390/390 [00:34<00:00, 11.41it/s, loss=6.267]


Validation perplexity ≈ 86.89

🔹 Seed:

Lords:
We have.

First Lord:
And grieve to hear't.


RuntimeError: Could not infer dtype of NoneType